In [1]:
import os
import pandas as pd
import numpy as np
from pandas import DataFrame, Series
from sklearn import linear_model



In [2]:
# Kaggle paths: prefer the provided leaf-classification folder, fallback to ../input for older notebooks
BASE_DIR_CANDIDATES = [
    "/kaggle/input/leaf-classification",
    "/kaggle/data/leaf-classification",
    "../input",
    "../input/leaf-classification",
]
BASE_DIR = None
for p in BASE_DIR_CANDIDATES:
    if os.path.exists(p) and (
        os.path.exists(os.path.join(p, "train.csv"))
        or os.path.exists(os.path.join(p, "leaf-classification", "train.csv"))
    ):
        BASE_DIR = p
        break


# Handle nested "leaf-classification/leaf-classification" cases
def _resolve_file(fname):
    if BASE_DIR is None:
        return fname
    direct = os.path.join(BASE_DIR, fname)
    nested = os.path.join(BASE_DIR, "leaf-classification", fname)
    if os.path.exists(direct):
        return direct
    if os.path.exists(nested):
        return nested
    return direct


train_path = _resolve_file("train.csv")
test_path = _resolve_file("test.csv")
sub_path = _resolve_file("sample_submission.csv")

train_df = pd.read_csv(train_path)
train_df.fillna(0, inplace=True)
train_df.head()



,id,species,margin1,margin2,margin3,margin4,margin5,margin6,margin7,margin8,...,texture55,texture56,texture57,texture58,texture59,texture60,texture61,texture62,texture63,texture64
0,186,Alnus_Cordata,0.001953,0.005859,0.039062,0.019531,0.025391,0.001953,0.058594,0.0,...,0.101560,0.000000,0.006836,0.000000,0.008789,0.000000,0.00000,0.008789,0.003906,0.023438
1,1152,Tilia_Platyphyllos,0.001953,0.011719,0.023438,0.033203,0.056641,0.003906,0.031250,0.0,...,0.007812,0.003906,0.002930,0.010742,0.046875,0.000977,0.00000,0.018555,0.008789,0.040039
2,435,Quercus_Imbricaria,0.037109,0.052734,0.031250,0.017578,0.003906,0.111330,0.029297,0.0,...,0.079102,0.003906,0.002930,0.000977,0.005859,0.000000,0.03125,0.004883,0.002930,0.035156
3,1467,Acer_Platanoids,0.007812,0.013672,0.072266,0.007812,0.007812,0.056641,0.003906,0.0,...,0.127930,0.041016,0.006836,0.003906,0.024414,0.000000,0.00000,0.017578,0.000000,0.025391
4,1222,Quercus_Semecarpifolia,0.027344,0.062500,0.060547,0.005859,0.001953,0.064453,0.029297,0.0,...,0.045898,0.000000,0.059570,0.003906,0.045898,0.000000,0.00000,0.002930,0.000000,0.034180


In [3]:
species = np.sort(train_df["species"].unique())
species_counts = len(species)
species_counts, species[:5]



(99,
 array(['Acer_Capillipes', 'Acer_Circinatum', 'Acer_Mono', 'Acer_Opalus',
        'Acer_Palmatum'], dtype=object))

In [4]:
# Encode species -> integers in a stable order
df = train_df.copy()
species_to_int = {s: i for i, s in enumerate(species)}
df["species"] = df["species"].map(species_to_int).astype(int)
df["species"].head()



0    10
1    93
2    67
3     6
4    79
Name: species, dtype: int64

In [5]:
# Fixes:
# - pandas.DataFrame.as_matrix deprecated -> use .to_numpy()
# - sklearn LogisticRegression needs an L1-capable solver; also use multinomial to get proper multiclass probabilities
# Core logic preserved: LogisticRegression with L1 penalty trained on provided features.
clf = linear_model.LogisticRegression(
    C=1.0,
    penalty="l1",
    tol=1e-6,
    solver="saga",
    multi_class="multinomial",
    max_iter=5000,
    random_state=0,
)

X = df.iloc[:, 2:].to_numpy(dtype=np.float64)  # features (exclude id, species)
y = df.iloc[:, 1].to_numpy(dtype=np.int64)  # encoded labels

clf.fit(X, y)



LogisticRegression(max_iter=5000, multi_class='multinomial', penalty='l1',
                   random_state=0, solver='saga', tol=1e-06)

In [6]:
# Confirm model trained
_ = clf.coef_.T
clf.coef_.shape, clf.intercept_.shape



((99, 192), (99,))

In [7]:
test_df = pd.read_csv(test_path)
test_df.fillna(0, inplace=True)

X_test = test_df.iloc[:, 1:].to_numpy(dtype=np.float64)  # exclude id
predict = clf.predict_proba(X_test)
predict.shape



(99, 99)

In [8]:
# Build submission aligned exactly to sample_submission.csv columns/order
sample_sub = pd.read_csv(sub_path)
sub_cols = list(sample_sub.columns)
species_cols = sub_cols[1:]  # exclude id

# Map probabilities (clf.classes_ are integer-encoded classes)
# Ensure all species columns exist and are in correct order
result = pd.DataFrame(
    0.0, index=np.arange(len(test_df)), columns=species_cols, dtype=np.float64
)

for class_int, probs in zip(clf.classes_, predict.T):
    class_name = species[class_int]
    if class_name in result.columns:
        result[class_name] = probs

result.insert(0, "id", test_df["id"].values)

# Probabilities must be in [0,1]; also avoid exact 0/1 for log-loss stability (Kaggle clamps, but safe to clip)
eps = 1e-15
for c in species_cols:
    result[c] = result[c].clip(eps, 1.0 - eps)

# Write to a valid .csv submission file in the working directory
out_path = "submission.csv"
result.to_csv(out_path, index=False)

out_path, result.shape, result.head()

('submission.csv',
 (99, 100),
      id  Acer_Capillipes  Acer_Circinatum  Acer_Mono  Acer_Opalus  \
 0  1202         0.009441         0.011801   0.009108     0.011801   
 1   992         0.009398         0.011748   0.009004     0.011748   
 2   491         0.009871         0.012339   0.009523     0.012339   
 3  1201         0.009907         0.012384   0.009692     0.012384   
 4    72         0.010181         0.012727   0.009926     0.012727   
 
    Acer_Palmatum  Acer_Pictum  Acer_Platanoids  Acer_Rubrum  Acer_Rufinerve  \
 0       0.003270     0.009441         0.011801     0.011484        0.009441   
 1       0.003908     0.009398         0.011748     0.002920        0.009398   
 2       0.003459     0.009871         0.012339     0.003483        0.009871   
 3       0.003592     0.009907         0.012384     0.003280        0.009907   
 4       0.003527     0.010181         0.012727     0.003164        0.010181   
 
    ...  Salix_Fragilis  Salix_Intergra  Sorbus_Aria  Tilia_Olive